# 1. Which is R1/R2?

#### In the Zhang_iScience_2022_Amel project, each read in _1.fastq and _2.fastq is 150bp in length, which does not conform to the format of "R1 contains barcode and UMI, R2 contains cDNA". Therefore, it is necessary to determine what each fastq contains, and which one is R1 and which one is R2.

#### I didn't thoroughly check whether each fastq contained UMI/barcode/cDNA. Instead, I directly ran `Cellranger count` using SRR15990871.lite.1_1.fastq.gz as R1 and SRR15990871.lite.1_2.fastq.gz as R2. The results are saved in /data/share/data/Zhou_lab_seq_data/20260401_lzy_sc_fastq/try_cellranger_count/SRR15990871/outs/web_summary.html

#### The final result shows 97.8% Valid Barcodes. Therefore, we will tentatively use<mark style="background: #FF5582A6;"> _1.fastq as R1 and _2.fastq as R2 in the next step.</mark>

# 2. The relation of Sample & lane

#### In this project, each caste (queen/nurse/forager) has two duplicates (GSM), each GSM has eight runs (SRR), and each SRR contains two fastqs. However, I don't know which one corresponds to `SampleName` in `<SampleName>_S<SampleNumber>_L00<Lane>_<Read>_001.fastq.gz`, and which one corresponds to `Lane`, in this data structure.

#### After discussion, it was temporarily agreed that<mark style="background: #FF5582A6;"> a single GSM would be considered a SampleName, but a single SRR would be considered a Lane.</mark>

#### Organize the correspondence between GSM and SRR into json format: 

In [ ]:
%%bash

# fetch the SRR name of GSM

while IFS= read -r GSM || [[ -n "$GSM" ]]; do # [[ -n "$GSM" ]] 防止文件最后一行没有换行符异常退出
    ffq "$GSM" > "./metadata/${GSM}.json"
done < "./metadata/all_GSM"

[2026-04-07 22:11:22,106]    INFO Parsing GSM GSM5590458
[2026-04-07 22:11:22,820]    INFO Finding supplementary files for GSM GSM5590458
[2026-04-07 22:11:27,383]    INFO Getting sample for GSM5590458
[2026-04-07 22:11:29,180]    INFO Parsing sample SRS10253564
[2026-04-07 22:11:30,291] WARNING Failed to parse sample information from ENA XML. Falling back to ENA search...
[2026-04-07 22:11:31,390]    INFO Getting Experiment for SRS10253564
[2026-04-07 22:11:31,390]    INFO Parsing Experiment SRX12279464
[2026-04-07 22:11:31,396] WARNING Failed to parse experiment information from ENA XML. Falling back to ENA search...
[2026-04-07 22:11:32,495] WARNING There are 8 runs for SRX12279464
[2026-04-07 22:11:32,495]    INFO Parsing run SRR15990911
[2026-04-07 22:11:36,707]    INFO Parsing run SRR15990913
[2026-04-07 22:11:40,924]    INFO Parsing run SRR15990914
[2026-04-07 22:11:46,966]    INFO Parsing run SRR15990917
[2026-04-07 22:11:51,178]    INFO Parsing run SRR15990912
[2026-04-07 22:1

In [13]:
import json

def GSM_SRR_relation(GSM_path):
    SRR = []
    with open(GSM_path, "r") as f:
        data = json.load(f)
    gsm = list(data.keys())[0]
    gsm_data = data[gsm]
    srp = None
    for sample in gsm_data["samples"].values():
        for exp in sample["experiments"].values():
            for run in exp["runs"].values():
                srr = run["accession"]
                SRR.append(srr)
    return gsm, SRR

all_GSM_SRR = {}

with open("./metadata/all_GSM", "r") as f:
    for GSM_name in f:
        GSM_name = GSM_name.rstrip()
        GSM_path = f"./metadata/{GSM_name}.json"
        all_GSM_SRR[GSM_SRR_relation(GSM_path)[0]] = GSM_SRR_relation(GSM_path)[1]

all_GSM_SRR


{'GSM5590458': ['SRR15990911',
  'SRR15990913',
  'SRR15990914',
  'SRR15990917',
  'SRR15990912',
  'SRR15990915',
  'SRR15990916',
  'SRR15990918'],
 'GSM5590457': ['SRR15990903',
  'SRR15990906',
  'SRR15990907',
  'SRR15990909',
  'SRR15990904',
  'SRR15990908',
  'SRR15990910',
  'SRR15990905'],
 'GSM5590456': ['SRR15990896',
  'SRR15990899',
  'SRR15990895',
  'SRR15990897',
  'SRR15990900',
  'SRR15990901',
  'SRR15990898',
  'SRR15990902'],
 'GSM5590455': ['SRR15990890',
  'SRR15990887',
  'SRR15990888',
  'SRR15990889',
  'SRR15990891',
  'SRR15990892',
  'SRR15990893',
  'SRR15990894'],
 'GSM5590454': ['SRR15990881',
  'SRR15990882',
  'SRR15990884',
  'SRR15990885',
  'SRR15990879',
  'SRR15990880',
  'SRR15990883',
  'SRR15990886'],
 'GSM5590453': ['SRR15990873',
  'SRR15990878',
  'SRR15990872',
  'SRR15990875',
  'SRR15990871',
  'SRR15990874',
  'SRR15990876',
  'SRR15990877']}

In [14]:
with open("./metadata/all_GSM_SRR.json", "w") as f:
    json.dump(all_GSM_SRR, f)

#### Based on the relationship between GSM and SRR, rebulid the Fastq file system (move the SRR fastq file into GSM directory): 

In [2]:
import json

with open("./metadata/all_GSM_SRR.json", "r") as f:
    all_GSM_SRR = json.load(f)

In [3]:
from pathlib import Path

for GSM_name in all_GSM_SRR:
    GSM_dir = Path(f"./fastq/{GSM_name}")
    if not GSM_dir.exists():
        GSM_dir.mkdir(parents=True, exist_ok=True)
    for SRR_name in all_GSM_SRR[GSM_name]:
        # print(SRR_name)
        matched_files = list(Path("./fastq").glob(f"{SRR_name}*.fastq.gz"))
        for SRR_file in matched_files:
            # print(SRR_file)
            target_SRR_file = GSM_dir / SRR_file.name
            SRR_file.rename(target_SRR_file)

#### Rename the fastq file into `<SampleName>_S<SampleNumber>_L00<Lane>_<Read>_001.fastq.gz` format

In [ ]:
%%bash

ls "./fastq/GSM5590453"

# SRR15990871.lite.1_1.fastq.gz
# SRR15990871.lite.1_2.fastq.gz
# SRR15990872.lite.1_1.fastq.gz
# SRR15990872.lite.1_2.fastq.gz
# SRR15990873.lite.1_1.fastq.gz
# SRR15990873.lite.1_2.fastq.gz
# SRR15990874.lite.1_1.fastq.gz
# SRR15990874.lite.1_2.fastq.gz
# SRR15990875.lite.1_1.fastq.gz
# SRR15990875.lite.1_2.fastq.gz
# SRR15990876.lite.1_1.fastq.gz
# SRR15990876.lite.1_2.fastq.gz
# SRR15990877.lite.1_1.fastq.gz
# SRR15990877.lite.1_2.fastq.gz
# SRR15990878.lite.1_1.fastq.gz
# SRR15990878.lite.1_2.fastq.gz

SRR15990871.lite.1_1.fastq.gz
SRR15990871.lite.1_2.fastq.gz
SRR15990872.lite.1_1.fastq.gz
SRR15990872.lite.1_2.fastq.gz
SRR15990873.lite.1_1.fastq.gz
SRR15990873.lite.1_2.fastq.gz
SRR15990874.lite.1_1.fastq.gz
SRR15990874.lite.1_2.fastq.gz
SRR15990875.lite.1_1.fastq.gz
SRR15990875.lite.1_2.fastq.gz
SRR15990876.lite.1_1.fastq.gz
SRR15990876.lite.1_2.fastq.gz
SRR15990877.lite.1_1.fastq.gz
SRR15990877.lite.1_2.fastq.gz
SRR15990878.lite.1_1.fastq.gz
SRR15990878.lite.1_2.fastq.gz


In [ ]:
for GSM_name in all_GSM_SRR:
    # print(GSM_name)
    value_list = [int(item[-2:]) for item in all_GSM_SRR[GSM_name]]
    base_value = min(value_list) - 1
    for SRR_name in all_GSM_SRR[GSM_name]:
        lane_value = int(SRR_name[-2:]) - base_value
        origin_name_R1 = Path(f"./fastq/{GSM_name}/{SRR_name}.lite.1_1.fastq.gz")
        origin_name_R2 = Path(f"./fastq/{GSM_name}/{SRR_name}.lite.1_2.fastq.gz")
        neo_name_R1 = Path(f"./fastq/{GSM_name}/{GSM_name}_S1_L00{lane_value}_R1_001.fastq.gz")
        neo_name_R2 = Path(f"./fastq/{GSM_name}/{GSM_name}_S1_L00{lane_value}_R2_001.fastq.gz")
        if origin_name_R1.exists() & origin_name_R2.exists():
            origin_name_R1.rename(neo_name_R1)
            origin_name_R2.rename(neo_name_R2)

# <SampleName>_S<SampleNumber>_L00<Lane>_<Read>_001.fastq.gz

GSM5590458
GSM5590457
GSM5590456
GSM5590455
GSM5590454
GSM5590453


In [8]:
%%bash

ls "./fastq/GSM5590453"

GSM5590453_S1_L001_R1_001.fastq.gz
GSM5590453_S1_L001_R2_001.fastq.gz
GSM5590453_S1_L002_R1_001.fastq.gz
GSM5590453_S1_L002_R2_001.fastq.gz
GSM5590453_S1_L003_R1_001.fastq.gz
GSM5590453_S1_L003_R2_001.fastq.gz
GSM5590453_S1_L004_R1_001.fastq.gz
GSM5590453_S1_L004_R2_001.fastq.gz
GSM5590453_S1_L005_R1_001.fastq.gz
GSM5590453_S1_L005_R2_001.fastq.gz
GSM5590453_S1_L006_R1_001.fastq.gz
GSM5590453_S1_L006_R2_001.fastq.gz
GSM5590453_S1_L007_R1_001.fastq.gz
GSM5590453_S1_L007_R2_001.fastq.gz
GSM5590453_S1_L008_R1_001.fastq.gz
GSM5590453_S1_L008_R2_001.fastq.gz
